# 同一 Qlib Recorder 的 DatasetH 回读修复

初次来源映射完成但 DatasetH 保存前缺少 dump_all 配置。只修复同一 Recorder 的 metadata/dataset 并回读，不重新读取来源或市场结果。

In [1]:
import os,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import pandas as pd,qlib
from qlib.workflow import R
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from src.data.nas_market_data import _settings,archive_nas_recorder
root=Path.cwd();card=root/'research/stage1_cftc_nq_tff_14d_positioning_map_20261023';repair=json.loads((card/'readback_repair_plan.json').read_text());plan=json.loads((card/'plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert repair['frozen'] and sha(card/'plan.json')==repair['original_plan_sha256'] and sha(card/'readback_failure.json')==repair['readback_failure_sha256'] and sha(card/'source_audit.json')==repair['source_audit_sha256']
tracking=card/'qlib_recorder';eid=repair['experiment_id'];rid=repair['recorder_id'];base=tracking/eid/rid/'artifacts';assert sha(base/'analysis/mapped_signals')==repair['mapped_artifact_sha256'] and sha(base/'metadata/dataset')==repair['old_dataset_artifact_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/plan['Qlib']['provider']),region='us',kernels=1)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid);mapped=rec.load_object('analysis/mapped_signals');source_audit=rec.load_object('analysis/source_audit')
assert len(mapped)==3964 and source_audit==json.loads((card/'source_audit.json').read_text())
calendar=pd.DatetimeIndex(mapped.index);assert calendar.is_monotonic_increasing and calendar.is_unique
static=mapped[['CFTC_NQ_LevNetOI_delta4','source_known']].copy();static.index=pd.MultiIndex.from_arrays([calendar,['SOXL_SOXX_CASH']*len(calendar)],names=['datetime','instrument']);static=pd.concat({'feature':static[['CFTC_NQ_LevNetOI_delta4']],'meta':static[['source_known']]},axis=1)
dataset=DatasetH(handler=DataHandlerLP.from_df(static),segments={'train':('2011-01-03','2022-12-30'),'held':('2023-01-03','2026-10-07')});assert len(dataset.prepare('train',col_set=['feature','meta']))==3020 and len(dataset.prepare('held',col_set=['feature','meta']))==944
dataset.config(dump_all=True,recursive=True)
with R.start(experiment_id=eid,recorder_id=rid,uri=str(tracking),resume=True):
 rec=R.get_recorder();assert rec.id==rid and str(rec.experiment_id)==eid;rec.save_objects(artifact_path='metadata',dataset=dataset)
with R.uri_context(str(tracking)):
 rec=R.get_recorder(recorder_id=rid,experiment_id=eid);saved=rec.load_object('metadata/dataset');assert len(saved.prepare('train',col_set=['feature','meta']))==3020 and len(saved.prepare('held',col_set=['feature','meta']))==944;assert sha(base/'analysis/mapped_signals')==repair['mapped_artifact_sha256']
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
evidence=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='cftc-map-repair-restore-') as td:
 archive=Path(td)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(evidence['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=120)
 assert run.returncode==0 and sha(archive)==evidence['sha256']
 with tarfile.open(archive,'r:gz') as z:names=[m.name for m in z.getmembers() if m.isfile()]
 assert any(x.endswith('/artifacts/analysis/mapped_signals') for x in names) and any(x.endswith('/artifacts/metadata/dataset') for x in names)
report=json.loads((card/'source_audit.json').read_text());summary={'experiment':plan['experiment'],'decision':report['status'],'objective_achieved':False,'plan':'plan.json','repair_plan':'readback_repair_plan.json','source_audit':'source_audit.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'mapped_artifact_sha256':repair['mapped_artifact_sha256'],'dataset_artifact_sha256':sha(base/'metadata/dataset'),'nas_evidence':evidence,'nas_readback':{'sha256':evidence['sha256'],'verified':True},'model_fits':0,'native_accounts':0,'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','metadata/*']}}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n');state=json.loads((card/'run_state.json').read_text());state.update(status='completed_after_same_recorder_metadata_repair',recorder_id=rid,nas_archive_sha256=evidence['sha256'],readback_repair_plan_sha256=sha(card/'readback_repair_plan.json'));(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n');print({'status':state['status'],'known_train':report['known_train_sessions'],'known_held':report['known_held_sessions'],'recorder_id':rid,'NAS':evidence['sha256']})


{'status': 'completed_after_same_recorder_metadata_repair', 'known_train': 2947, 'known_held': 915, 'recorder_id': 'cdd3e57a851a48d8bc563bac302084f5', 'NAS': '3b7c83dd352b5b7445375730231b3c912af657da876798f580d30297babb417a'}


## 中文结论

在原 Recorder 中补存 DatasetH 并通过 3020/944 行回读；映射对象哈希未变，NAS 远端归档和指定成员恢复通过。该来源仍只准开发期信息筛选，见 `conclusion.md`。